In [3]:
import pandas as pd
import numpy as np
from datetime import datetime

# 1. Load Data
df = pd.read_csv(r'./data/2026-04-06/option_chain_nifty_2026-04-07_full_history.csv')

# 2. Basic Cleaning
df['fetch_time'] = pd.to_datetime(df['fetch_time'])
df['expiry'] = pd.to_datetime(df['expiry'])
df.drop_duplicates(subset=['fetch_time', 'strike_price'], inplace=True)

# 3. Parameters
r = 0.07  # Assume 7% risk-free rate
# Calculate T in years (seconds remaining / seconds in a year)
df['T'] = (df['expiry'] - df['fetch_time']).dt.total_seconds() / (365 * 24 * 3600)

# 4. Compute Parity Components
# LHS = C - P
df['LHS'] = df['ce_liveData_ltp'] - df['pe_liveData_ltp']

# RHS = S - K * exp(-rt)
df['RHS'] = df['spot_price'] - (df['strike_price'] * np.exp(-r * df['T']))

# 5. Identify Deviations
# Deviation = LHS - RHS
df['deviation'] = df['LHS'] - df['RHS']

# 6. Filter for "Liquidity" to make it realistic
# Only look at rows where markers aren't 'ILLIQUID'
df_liquid = df[~df['ce_markers'].str.contains('ILLIQUID') & ~df['pe_markers'].str.contains('ILLIQUID')].copy()

# Output potential arbitrage opportunities
# Positive deviation: Call is overpriced / Put is underpriced
# Negative deviation: Put is overpriced / Call is underpriced
opportunities = df_liquid[abs(df_liquid['deviation']) > 5] # Filtering for significant deviations > 5 points
print(opportunities[['fetch_time', 'strike_price', 'deviation']].head(20))

            fetch_time  strike_price  deviation
8  2026-04-06 09:16:00       20500.0  -6.263366
26 2026-04-06 09:16:00       21400.0  29.880681
28 2026-04-06 09:16:00       21500.0  -5.781091
29 2026-04-06 09:16:00       21550.0  53.363022
30 2026-04-06 09:16:00       21600.0  29.507136
32 2026-04-06 09:16:00       21700.0  17.245363
33 2026-04-06 09:16:00       21750.0  20.139477
35 2026-04-06 09:16:00       21850.0  22.977705
36 2026-04-06 09:16:00       21900.0 -18.778182
37 2026-04-06 09:16:00       21950.0 -17.834068
38 2026-04-06 09:16:00       22000.0 -14.889954
39 2026-04-06 09:16:00       22050.0 -19.045840
40 2026-04-06 09:16:00       22100.0 -14.401727
41 2026-04-06 09:16:00       22150.0 -15.607613
42 2026-04-06 09:16:00       22200.0 -15.613499
43 2026-04-06 09:16:00       22250.0 -17.319385
44 2026-04-06 09:16:00       22300.0 -14.875272
45 2026-04-06 09:16:00       22350.0 -11.331158
46 2026-04-06 09:16:00       22400.0 -13.937044
47 2026-04-06 09:16:00       22450.0 -12

In [6]:
# Assuming 'df' already has the 'deviation' column from the previous step

# 1. Sort by absolute deviation (Largest mispricing first)
# This is best for identifying the "best" arbitrage overall
df['abs_deviation'] = df['deviation'].abs()
best_opportunities = df.sort_values(by='abs_deviation', ascending=False)

# 2. Sort by raw deviation (Directional)
# Highest values = Best for 'Conversion' trades
# Lowest (most negative) values = Best for 'Reversal' trades
conversion_deals = df.sort_values(by='deviation', ascending=False)
reversal_deals = df.sort_values(by='deviation', ascending=True)

# 3. Filter out noise
# Even if a derivation is large, if volume is 0, you can't trade it.
# ce_liveData_oi is Open Interest (liquidity indicator)
tradable_sort = df[(df['ce_liveData_oi'] > 0) & (df['pe_liveData_oi'] > 0)].sort_values(by='abs_deviation', ascending=False)

print(tradable_sort[['strike_price', 'deviation', 'ce_liveData_oi']].head(20))

        strike_price   deviation  ce_liveData_oi
445657       26400.0  645.197638            2753
445526       26400.0  645.197520            2753
445395       26400.0  645.197403            2753
445264       26400.0  645.197286            2753
445133       26400.0  645.197169            2753
445002       26400.0  645.197052            2753
444871       26400.0  645.196934            2753
444740       26400.0  645.196817            2753
444609       26400.0  645.196700            2753
444478       26400.0  645.196583            2753
444347       26400.0  645.196466            2753
444216       26400.0  645.196348            2753
444085       26400.0  645.196231            2753
443954       26400.0  645.196114            2753
443823       26400.0  645.195997            2753
443692       26400.0  645.195880            2753
447753       26400.0  638.849513            2753
447622       26400.0  638.849395            2753
447491       26400.0  638.849278            2753
447360       26400.0

In [5]:
print(tradable_sort[['strike_price', 'deviation', 'ce_liveData_oi']].tail(10))

         strike_price  deviation  ce_liveData_oi
1244880       26000.0   0.000337           20793
539903        22700.0  -0.000317          125034
540689        22700.0   0.000288          125034
157270        23600.0  -0.000249           53664
540034        22700.0  -0.000216          125034
540558        22700.0   0.000187          125034
157401        23600.0  -0.000144           53664
540165        22700.0  -0.000115          125034
540427        22700.0   0.000086          125034
540296        22700.0  -0.000014          125034
